# Resource Cleanup

This notebook helps you clean up all AWS resources created during the workshop to avoid unnecessary charges. We'll clean up the following resources:

1. SageMaker endpoints
2. SageMaker models
3. SageMaker training jobs
4. S3 objects
5. CloudWatch logs

**Important**: Make sure you've completed all the workshop exercises before running this cleanup notebook.

In [ ]:
# Install required packages
!pip install -q "boto3>=1.35.0" "sagemaker>=2.230.0" "pandas>=2.2.0" "tqdm>=4.66.0"

In [ ]:
# Import required libraries
import os
import boto3
import sagemaker
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime, timedelta

# Set up AWS clients
sagemaker_client = boto3.client('sagemaker')
s3_client = boto3.client('s3')
s3_resource = boto3.resource('s3')
logs_client = boto3.client('logs')

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()
prefix = "fine-tuning-workshop"

print(f"Using S3 bucket: {bucket}")

## 1. List Resources

First, let's list all the resources that were created during the workshop.

In [ ]:
# Function to list SageMaker endpoints
def list_endpoints():
    endpoints = []
    next_token = None
    
    while True:
        if next_token:
            response = sagemaker_client.list_endpoints(NextToken=next_token, MaxResults=100)
        else:
            response = sagemaker_client.list_endpoints(MaxResults=100)
        
        for endpoint in response['Endpoints']:
            # Filter endpoints created for this workshop
            if 'fine-tuned-sentiment' in endpoint['EndpointName'] or 'quantized' in endpoint['EndpointName'] or 'pruned' in endpoint['EndpointName'] or 'distilled' in endpoint['EndpointName']:
                endpoints.append({
                    'EndpointName': endpoint['EndpointName'],
                    'CreationTime': endpoint['CreationTime'],
                    'EndpointStatus': endpoint['EndpointStatus']
                })
        
        if 'NextToken' in response:
            next_token = response['NextToken']
        else:
            break
    
    return endpoints

# Function to list SageMaker models
def list_models():
    models = []
    next_token = None
    
    while True:
        if next_token:
            response = sagemaker_client.list_models(NextToken=next_token, MaxResults=100)
        else:
            response = sagemaker_client.list_models(MaxResults=100)
        
        for model in response['Models']:
            # Filter models created for this workshop
            if 'fine-tuned' in model['ModelName'] or 'quantized' in model['ModelName'] or 'pruned' in model['ModelName'] or 'distilled' in model['ModelName']:
                models.append({
                    'ModelName': model['ModelName'],
                    'CreationTime': model['CreationTime']
                })
        
        if 'NextToken' in response:
            next_token = response['NextToken']
        else:
            break
    
    return models

# Function to list SageMaker training jobs
def list_training_jobs():
    training_jobs = []
    next_token = None
    
    while True:
        if next_token:
            response = sagemaker_client.list_training_jobs(NextToken=next_token, MaxResults=100)
        else:
            response = sagemaker_client.list_training_jobs(MaxResults=100)
        
        for job in response['TrainingJobSummaries']:
            # Filter training jobs created for this workshop
            if 'fine-tuning' in job['TrainingJobName'] or 'quantization' in job['TrainingJobName'] or 'pruning' in job['TrainingJobName'] or 'distillation' in job['TrainingJobName']:
                training_jobs.append({
                    'TrainingJobName': job['TrainingJobName'],
                    'CreationTime': job['CreationTime'],
                    'TrainingJobStatus': job['TrainingJobStatus']
                })
        
        if 'NextToken' in response:
            next_token = response['NextToken']
        else:
            break
    
    return training_jobs

# Function to list S3 objects in the workshop prefix
def list_s3_objects():
    objects = []
    paginator = s3_client.get_paginator('list_objects_v2')
    
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                objects.append({
                    'Key': obj['Key'],
                    'Size': obj['Size'],
                    'LastModified': obj['LastModified']
                })
    
    return objects

# List all resources
print("Listing SageMaker endpoints...")
endpoints = list_endpoints()
print(f"Found {len(endpoints)} endpoints")

print("\nListing SageMaker models...")
models = list_models()
print(f"Found {len(models)} models")

print("\nListing SageMaker training jobs...")
training_jobs = list_training_jobs()
print(f"Found {len(training_jobs)} training jobs")

print("\nListing S3 objects...")
s3_objects = list_s3_objects()
print(f"Found {len(s3_objects)} S3 objects")

# Display resources in DataFrames
if endpoints:
    print("\nSageMaker Endpoints:")
    display(pd.DataFrame(endpoints))

if models:
    print("\nSageMaker Models:")
    display(pd.DataFrame(models))

if training_jobs:
    print("\nSageMaker Training Jobs:")
    display(pd.DataFrame(training_jobs))

if s3_objects:
    print(f"\nS3 Objects in {bucket}/{prefix}:")
    # Show only the first 10 objects if there are many
    s3_df = pd.DataFrame(s3_objects)
    if len(s3_df) > 10:
        display(s3_df.head(10))
        print(f"...and {len(s3_df) - 10} more objects")
    else:
        display(s3_df)

## 2. Clean Up Resources

Now, let's clean up all the resources we found.

In [ ]:
# Function to delete SageMaker endpoints
def delete_endpoints(endpoints):
    for endpoint in tqdm(endpoints, desc="Deleting endpoints"):
        try:
            sagemaker_client.delete_endpoint(EndpointName=endpoint['EndpointName'])
            print(f"Deleted endpoint: {endpoint['EndpointName']}")
        except Exception as e:
            print(f"Error deleting endpoint {endpoint['EndpointName']}: {e}")

# Function to delete SageMaker models
def delete_models(models):
    for model in tqdm(models, desc="Deleting models"):
        try:
            sagemaker_client.delete_model(ModelName=model['ModelName'])
            print(f"Deleted model: {model['ModelName']}")
        except Exception as e:
            print(f"Error deleting model {model['ModelName']}: {e}")

# Function to delete S3 objects
def delete_s3_objects(bucket, objects):
    if not objects:
        return
    
    # Delete objects in batches of 1000 (S3 limit)
    for i in tqdm(range(0, len(objects), 1000), desc="Deleting S3 objects"):
        batch = objects[i:i+1000]
        try:
            s3_client.delete_objects(
                Bucket=bucket,
                Delete={
                    'Objects': [{'Key': obj['Key']} for obj in batch],
                    'Quiet': True
                }
            )
            print(f"Deleted {len(batch)} objects from S3")
        except Exception as e:
            print(f"Error deleting S3 objects: {e}")

# Function to delete CloudWatch log groups for training jobs
def delete_log_groups(training_jobs):
    log_groups = []
    
    # Get log groups for training jobs
    for job in training_jobs:
        log_group_name = f"/aws/sagemaker/TrainingJobs/{job['TrainingJobName']}"
        log_groups.append(log_group_name)
    
    # Delete log groups
    for log_group in tqdm(log_groups, desc="Deleting log groups"):
        try:
            logs_client.delete_log_group(logGroupName=log_group)
            print(f"Deleted log group: {log_group}")
        except Exception as e:
            print(f"Error deleting log group {log_group}: {e}")

### 2.1 Delete SageMaker Endpoints

First, let's delete all SageMaker endpoints to stop incurring charges for running instances.

In [ ]:
# Delete SageMaker endpoints
if endpoints:
    print(f"Deleting {len(endpoints)} SageMaker endpoints...")
    delete_endpoints(endpoints)
    print("Endpoints deleted successfully.")
else:
    print("No endpoints to delete.")

### 2.2 Delete SageMaker Models

Now, let's delete all SageMaker models.

In [ ]:
# Delete SageMaker models
if models:
    print(f"Deleting {len(models)} SageMaker models...")
    delete_models(models)
    print("Models deleted successfully.")
else:
    print("No models to delete.")

### 2.3 Delete CloudWatch Logs

Let's delete the CloudWatch logs associated with our training jobs.

In [ ]:
# Delete CloudWatch logs
if training_jobs:
    print(f"Deleting CloudWatch logs for {len(training_jobs)} training jobs...")
    delete_log_groups(training_jobs)
    print("Log groups deleted successfully.")
else:
    print("No training jobs to delete logs for.")

### 2.4 Delete S3 Objects

Finally, let's delete all S3 objects created during the workshop.

In [ ]:
# Delete S3 objects
if s3_objects:
    print(f"Deleting {len(s3_objects)} S3 objects...")
    delete_s3_objects(bucket, s3_objects)
    print("S3 objects deleted successfully.")
else:
    print("No S3 objects to delete.")

## 3. Verify Cleanup

Let's verify that all resources have been cleaned up properly.

In [ ]:
# Verify cleanup
print("Verifying cleanup...")

# Check endpoints
remaining_endpoints = list_endpoints()
if remaining_endpoints:
    print(f"Warning: {len(remaining_endpoints)} endpoints still exist:")
    for endpoint in remaining_endpoints:
        print(f"  - {endpoint['EndpointName']} (Status: {endpoint['EndpointStatus']})")
else:
    print("✓ All endpoints deleted successfully.")

# Check models
remaining_models = list_models()
if remaining_models:
    print(f"Warning: {len(remaining_models)} models still exist:")
    for model in remaining_models:
        print(f"  - {model['ModelName']}")
else:
    print("✓ All models deleted successfully.")

# Check S3 objects
remaining_objects = list_s3_objects()
if remaining_objects:
    print(f"Warning: {len(remaining_objects)} S3 objects still exist in {bucket}/{prefix}")
else:
    print(f"✓ All S3 objects in {bucket}/{prefix} deleted successfully.")

## 4. Additional Cleanup Steps

If you're completely done with the workshop, you may want to consider these additional cleanup steps:

1. **SageMaker Notebook Instance**: If you're using a SageMaker notebook instance, you should stop or delete it to avoid incurring charges.
2. **CloudFormation Stack**: If you used CloudFormation to set up the workshop environment, you should delete the stack.
3. **IAM Roles**: If you created any IAM roles specifically for this workshop, you may want to delete them.

These steps are optional and can be performed through the AWS Management Console.

## 5. Conclusion

You have successfully cleaned up all AWS resources created during the Model Optimization Workshop. This will help you avoid unnecessary charges on your AWS account.

Thank you for participating in the workshop! We hope you found it informative and valuable.